# Concrete QC project - make an anonymised copy of the data

Goal: create versions of the data that are safe to put on GitHub.
The raw data has project details (LR numbers, lab, supplier, places), so we hide them here.
Run this file once from VS Code. The new files go into data/anonymised.

### Step 1: Import the libraries and find the folders

In [ ]:
from pathlib import Path
import pandas as pd

project = Path.cwd().parent   # the notebook runs from the notebooks folder, so the project is one level up

raw = project / "data" / "raw"
anonymised = project / "data" / "anonymised"
anonymised.mkdir(exist_ok=True)

### Step 2: Load the raw files

In [ ]:
cubes = pd.read_csv(raw / "cube_results.csv")
mixes = pd.read_csv(raw / "mix_designs.csv")
durability = pd.read_csv(raw / "durability_results.csv")

### Step 3: Give every pour (LR number) a new ID

LR-0481 in the cubes file and LR-481 in the durability file are the same pour,
so we keep only the number first, then give each number one new ID (POUR-0001, POUR-0002 ...).
This way the two files can still be joined later.

In [ ]:
cubes["lr_number"] = cubes["lr"].str.extract(r"(\d+)").astype(float)
durability["lr_number"] = durability["lr"].str.extract(r"(\d+)").astype(float)

all_numbers = sorted(set(cubes["lr_number"].dropna()) | set(durability["lr_number"].dropna()))
new_id = {}
for i, number in enumerate(all_numbers, start=1):
    new_id[number] = "POUR-" + str(i).zfill(4)

cubes["pour_id"] = cubes["lr_number"].map(new_id)
durability["pour_id"] = durability["lr_number"].map(new_id)

### Step 4: Move all the dates by the same number of days

The gaps between dates (and the 7 / 28 day ages) stay correct, but the real dates are hidden.
Keep this number to yourself.

In [ ]:
shift_days = -371
cubes["date_cast"] = pd.to_datetime(cubes["date_cast"], errors="coerce") + pd.Timedelta(days=shift_days)
durability["date_cast"] = pd.to_datetime(durability["date_cast"], errors="coerce") + pd.Timedelta(days=shift_days)

### Step 5: Hide names

Mix IDs start with AFR- (the supplier), so we change it to MIX-

In [ ]:
cubes["mix_id"] = cubes["mix_id"].str.replace("AFR-", "MIX-")
mixes["mix_id"] = mixes["mix_id"].str.replace("AFR-", "MIX-")
mixes["supplier"] = "Supplier A"

# The folder column looks like "03_Structures/Ramp A/Piers". We only keep the last part ("Piers").
cubes["element_category"] = cubes["folder"].str.split("/").str[-1]

# Structure names (Ramp A, Sky Circle ...) become Structure 1, Structure 2 ...
structures = sorted(durability["structure"].dropna().unique())
structure_id = {}
for i, name in enumerate(structures, start=1):
    structure_id[name] = "Structure " + str(i)
durability["structure"] = durability["structure"].map(structure_id)
durability["element_type"] = durability["element_type"].str.replace("Deck (Sky Circle)", "Deck", regex=False)

# The flags column mentions real dates and the lab's name, so we clean it too
durability["flags"] = durability["flags"].fillna("")
durability["flags"] = durability["flags"].str.replace(r"\d{4}-\d{2}-\d{2}", "<date>", regex=True)
durability["flags"] = durability["flags"].str.replace("SWL", "the lab")

### Step 6: Drop the columns that give the project away

In [ ]:
cubes = cubes.drop(columns=["lr", "lr_number", "swl", "supplier_report", "element", "folder"])
mixes = mixes.drop(columns=["d2_source"])
durability = durability.drop(columns=["lr", "lr_number", "swl", "panel", "element", "report_date", "source", "file"])

### Step 7: Save the anonymised files

In [ ]:
cubes.to_csv(anonymised / "cube_results_anon.csv", index=False)
mixes.to_csv(anonymised / "mix_designs_anon.csv", index=False)
durability.to_csv(anonymised / "durability_results_anon.csv", index=False)

print("Saved to:", anonymised)
print("Cubes:", cubes.shape, " Mixes:", mixes.shape, " Durability:", durability.shape)

### Step 8: Double check nothing is left behind

These words should not appear anywhere in the anonymised files. We expect 0 for each.

In [ ]:
words = ["AfriSam", "Afrisam", "Ramp", "Sky Circle", "LR-", "SWL", "Govan", "Steyn"]
for name in ["cube_results_anon.csv", "mix_designs_anon.csv", "durability_results_anon.csv"]:
    text = (anonymised / name).read_text(encoding="utf-8")
    found = [w for w in words if w in text]
    print(name, "- words found:", found)